# Lab 1: Project, identity and chat inference

**Prerequisites:** Administrator-approved Foundry account, network, chat quota and permissions.
**Recommended route:** one-day Labs 1 -> 4 -> 7.
Establish approved identity/project scope, deploy only a chat model and verify inference.
No embedding model or RAG setup. Hosting enablement and roles remain administrator-owned.
Provisioning and inference incur charges; review each explicit approval before acting.
Save scope-bound chat evidence for Lab 4; never fabricate acceptance or broaden permissions.
Use a fresh Python 3.14 dev-container kernel; run all cells in order.

This cell locates the repository and imports the Azure CLI provisioning helpers.

In [ ]:
# Step 1.1 - Import project helpers
import json
from datetime import datetime, timezone
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
REPO_ROOT = next(p for p in (HERE, *HERE.parents)
                 if (p / "build-and-operate-foundry-agents/common/resource_names.py").is_file())
WORKSHOP = REPO_ROOT / "build-and-operate-foundry-agents"
LAB_DIR = WORKSHOP / "shared/foundry-project-models"
for folder in (WORKSHOP, LAB_DIR, WORKSHOP / "1-day-labs"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from common import resource_names
import one_day_parts as notebook_parts
import project_setup
from common import foundry_env

CLI = project_setup.AzureCLI()
ARTIFACTS = WORKSHOP / "1-day-labs/artifacts/lab1"
for name in ("part_a.json", "part_b.json", "project.json"):
    (ARTIFACTS / name).unlink(missing_ok=True)

This cell records explicit approved identity inputs without switching the active Azure subscription.

In [ ]:
# Step 1.2 - Edit approved context
SUBSCRIPTION_ID = ""  # Blank initially discovers the active subscription; fill after review.
TENANT_ID = ""  # Fill the approved tenant GUID after discovery.
RESOURCE_GROUP = ""  # Existing facilitator-approved resource group.
FOUNDRY_ACCOUNT_NAME = ""  # Existing Foundry account, never a new account.
ATTENDEE_SUFFIX = ""  # Unique 1-16 lowercase letters/numbers/hyphens.
APPROVE_PROJECT = False  # Set True only after reviewing account, scope and ownership.

This cell displays the authenticated identity context and project-enabled existing accounts for facilitator review.

In [ ]:
# Step 1.3 - Discover identity and accounts
for name in ("part_a.json", "part_b.json", "project.json"):
    (ARTIFACTS / name).unlink(missing_ok=True)
CONTEXT = project_setup.azure_context(CLI, SUBSCRIPTION_ID, TENANT_ID)
print(json.dumps(CONTEXT, indent=2))
print(json.dumps(project_setup.account_inventory(CLI, CONTEXT), indent=2))
print("Fill both approved GUIDs and the selected account in Step 1.2, then rerun it.")

This cell verifies the selected account and prints the attendee-owned project scope before any writes.

In [ ]:
# Step 1.4 - Review project ownership
for name in ("part_a.json", "part_b.json", "project.json"):
    (ARTIFACTS / name).unlink(missing_ok=True)
if not SUBSCRIPTION_ID or not TENANT_ID:
    raise ValueError("Enter the reviewed subscription and tenant GUIDs in Step 1.2.")
CONTEXT = project_setup.azure_context(CLI, SUBSCRIPTION_ID, TENANT_ID)
SUFFIX = resource_names.suffix({"MARKETPLACE_RESOURCE_SUFFIX": ATTENDEE_SUFFIX}, required=True)
ACCOUNT = project_setup.verify_account(CLI, CONTEXT, RESOURCE_GROUP, FOUNDRY_ACCOUNT_NAME)
print(json.dumps({"context": CONTEXT, "account_id": ACCOUNT["id"],
                  "project_name": resource_names.name("healthcare-marketplace",
                      {"MARKETPLACE_RESOURCE_SUFFIX": SUFFIX}, required=True)}, indent=2))

This cell creates or safely reuses only your owned project and publishes a narrowly scoped noncredential handoff.

In [ ]:
# Step 1.5 - Create project and save handoff
for name in ("part_a.json", "part_b.json", "project.json"):
    (ARTIFACTS / name).unlink(missing_ok=True)
if APPROVE_PROJECT is not True:
    raise RuntimeError("Review Step 1.4 and set APPROVE_PROJECT=True in Step 1.2.")
CONTEXT = project_setup.azure_context(CLI, SUBSCRIPTION_ID, TENANT_ID)
ACCOUNT = project_setup.verify_account(CLI, CONTEXT, RESOURCE_GROUP, FOUNDRY_ACCOUNT_NAME)
PROJECT = project_setup.ensure_project(CLI, ACCOUNT, SUFFIX)
PROJECT_ENDPOINT, OPENAI_ENDPOINT = project_setup.endpoints(ACCOUNT, PROJECT)
HANDOFF = {
    "schema_version": 1, "part": "a", "verified_at": datetime.now(timezone.utc).isoformat(),
    "subscription_id": CONTEXT["subscription_id"], "tenant_id": CONTEXT["tenant_id"],
    "resource_group": RESOURCE_GROUP, "account_name": FOUNDRY_ACCOUNT_NAME,
    "account_resource_id": ACCOUNT["id"], "project_resource_id": PROJECT["id"],
    "resource_suffix": SUFFIX, "project_endpoint": PROJECT_ENDPOINT,
    "azure_openai_endpoint": OPENAI_ENDPOINT,
}
PROJECT_CONTEXT = {
    "subscription_id": CONTEXT["subscription_id"].lower(), "tenant_id": CONTEXT["tenant_id"].lower(),
    "account_resource_id": ACCOUNT["id"].lower(), "project_resource_id": PROJECT["id"].lower(),
    "resource_suffix": SUFFIX,
}
notebook_parts.write_checkpoint(
    ARTIFACTS / "part_a.json", lab="lab1", part="a", context=PROJECT_CONTEXT, state=HANDOFF)
print(json.dumps(HANDOFF, indent=2))

The project is now established. Confirm hosted-agent enablement and identity
access with the administrator; a chat smoke test does not verify hosting.
This cell discovers chat model versions and advertised SKUs without cloud writes.

In [ ]:
# Step 1.6 - Discover chat models
MODELS = CLI.items(ACCOUNT["id"] + "/models")
print(json.dumps([m for m in MODELS if m.get("format") == "OpenAI"
                  and m.get("name", "").startswith("gpt-")], indent=2))

Review the advertised model/version, quota, SKU and cost before approving deployment.
This cell selects only a chat deployment; the one-day track performs no embedding or RAG setup.

In [ ]:
# Step 1.7 - Review the chat deployment plan
CHAT_MODEL = "gpt-5.4-mini"
CHAT_VERSION = ""
CHAT_SKU = "GlobalStandard"
CHAT_CAPACITY = 10  # Review shared room capacity with the administrator.
APPROVE_PROVISIONING = False
CHAT_SPEC = project_setup.choose_model(MODELS, CHAT_MODEL, CHAT_VERSION, CHAT_SKU, CHAT_CAPACITY)
CHAT_NAME = resource_names.name("marketplace-chat", {"MARKETPLACE_RESOURCE_SUFFIX": SUFFIX}, required=True)
SMOKE_TESTS = {}
SMOKE_TARGET = None
print(json.dumps({"account_id": ACCOUNT["id"], "chat": {"name": CHAT_NAME, **CHAT_SPEC}}, indent=2))

This cell rechecks approved identity and project ownership and explicitly provisions only the selected chat model.

In [ ]:
# Step 1.8 - Deploy the chat model
SMOKE_TESTS = {}
SMOKE_TARGET = None
ARTIFACT = ARTIFACTS / "project.json"
ARTIFACT.unlink(missing_ok=True)
(ARTIFACTS / "part_b.json").unlink(missing_ok=True)
if APPROVE_PROVISIONING is not True:
    raise RuntimeError("Review Step 1.7 and set APPROVE_PROVISIONING=True before proceeding.")
HANDOFF, CONTEXT, ACCOUNT, PROJECT = project_setup.read_project_handoff(
    ARTIFACTS / "part_a.json", CLI, SUBSCRIPTION_ID, TENANT_ID, ATTENDEE_SUFFIX)
PROJECT_ENDPOINT, OPENAI_ENDPOINT = project_setup.endpoints(ACCOUNT, PROJECT)
CHAT = project_setup.ensure_deployment(CLI, ACCOUNT, CHAT_NAME, CHAT_SPEC)
print(json.dumps({"project_id": PROJECT["id"], "chat": CHAT["name"]}, indent=2))

This cell checks real chat inference with Entra authentication and binds the evidence to the exact deployment plan.

In [ ]:
# Step 1.9 - Verify chat inference
SMOKE_TESTS = {}
SMOKE_TARGET = None
ARTIFACT.unlink(missing_ok=True)
(ARTIFACTS / "part_b.json").unlink(missing_ok=True)
project_setup.read_project_handoff(
    ARTIFACTS / "part_a.json", CLI, SUBSCRIPTION_ID, TENANT_ID, ATTENDEE_SUFFIX)
TESTED_TARGET = json.dumps({
    "project_id": PROJECT["id"], "project_endpoint": PROJECT_ENDPOINT,
    "openai_endpoint": OPENAI_ENDPOINT, "tenant_id": CONTEXT["tenant_id"],
    "chat": {"name": CHAT_NAME, "spec": CHAT_SPEC},
}, sort_keys=True)
reply = CLI.run("rest", "--method", "post", "--resource", "https://cognitiveservices.azure.com",
                "--url", OPENAI_ENDPOINT + "openai/v1/chat/completions",
                "--body", json.dumps({"model": CHAT_NAME, "messages": [
                    {"role": "user", "content": "Say hello in one short sentence."}]}))
choices = reply.get("choices", [])
text = choices[0].get("message", {}).get("content") if choices else None
if not isinstance(text, str) or not text.strip():
    raise RuntimeError("Chat smoke test returned no assistant text.")
print(text)
SMOKE_TESTS = {"chat": "passed"}
SMOKE_TARGET = TESTED_TARGET

This cell publishes chat-only configuration and scoped evidence only after matching the current live project and tested plan.

In [ ]:
# Step 1.10 - Publish verified setup
ARTIFACT.unlink(missing_ok=True)
(ARTIFACTS / "part_b.json").unlink(missing_ok=True)
HANDOFF, CONTEXT, ACCOUNT, PROJECT = project_setup.read_project_handoff(
    ARTIFACTS / "part_a.json", CLI, SUBSCRIPTION_ID, TENANT_ID, ATTENDEE_SUFFIX)
PROJECT_ENDPOINT, OPENAI_ENDPOINT = project_setup.endpoints(ACCOUNT, PROJECT)
CURRENT_TARGET = json.dumps({
    "project_id": PROJECT["id"], "project_endpoint": PROJECT_ENDPOINT,
    "openai_endpoint": OPENAI_ENDPOINT, "tenant_id": CONTEXT["tenant_id"],
    "chat": {"name": CHAT_NAME, "spec": CHAT_SPEC},
}, sort_keys=True)
if SMOKE_TESTS.get("chat") != "passed" or SMOKE_TARGET != CURRENT_TARGET:
    raise RuntimeError("Run Step 1.9 for this exact project and chat deployment before publishing.")
OUTPUT_ENV = {
    "FOUNDRY_PROJECT_ENDPOINT": PROJECT_ENDPOINT, "PROJECT_RESOURCE_ID": PROJECT["id"],
    "TENANT_ID": CONTEXT["tenant_id"], "AZURE_AI_MODEL_DEPLOYMENT_NAME": CHAT_NAME,
    "FOUNDRY_MODEL": CHAT_NAME, "AZURE_OPENAI_ENDPOINT": OPENAI_ENDPOINT,
    "MARKETPLACE_RESOURCE_SUFFIX": SUFFIX,
}
project_setup.write_env(REPO_ROOT / ".env", OUTPUT_ENV)
ENV = foundry_env.load_env()
if any(ENV.get(key) != value for key, value in OUTPUT_ENV.items()):
    raise RuntimeError("Downstream configuration does not match the verified setup.")
CHECKPOINT = {
    "schema_version": 1, "track": "one-day-chat-only",
    "verified_at": datetime.now(timezone.utc).isoformat(),
    "subscription_id": CONTEXT["subscription_id"], "tenant_id": CONTEXT["tenant_id"],
    "account_resource_id": ACCOUNT["id"], "project_resource_id": PROJECT["id"],
    "project_endpoint": PROJECT_ENDPOINT, "azure_openai_endpoint": OPENAI_ENDPOINT,
    "resource_suffix": SUFFIX, "chat_deployment": {"name": CHAT_NAME, **CHAT_SPEC},
    "provisioning_state": "Succeeded", "smoke_tests": SMOKE_TESTS,
}
foundry_env.save_artifact(ARTIFACT, CHECKPOINT)
notebook_parts.write_checkpoint(
    ARTIFACTS / "part_b.json", lab="lab1", part="b", context=notebook_parts.scope(ENV),
    evidence=[ARTIFACT], state={"track": "one-day-chat-only", "smoke_tests": SMOKE_TESTS,
                              "project_resource_id": PROJECT["id"]})
print("Lab 1 complete: project and chat inference verified. Continue with Lab 4.")